# Step 11: Multi-Crop Feature Engineering
## MandiMitra ML Pipeline — Tomato, Wheat, Cotton

Engineer lag, rolling, momentum, spread, and calendar features independently per series. Zero leakage.

### 1. Setup

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
BASE_DIR = Path('..').resolve()
if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))
from src.multi_crop_feature_engineering import engineer_all_crops
print('Step 11: Feature Engineering')

Step 11: Feature Engineering


### 2. Engineer Features for All Crops

In [2]:
feat_results = engineer_all_crops(verbose=True)


FEATURE ENGINEERING: TOMATO
  APMC Kamthi | Local | Local: 510 obs -> 507 valid 3-obs targets
  APMC Panvel | Other | Local: 527 obs -> 524 valid 3-obs targets
  Pune(Manjri) | Other | Local: 539 obs -> 536 valid 3-obs targets
  Pune(Pimpri) | Local | Local: 550 obs -> 547 valid 3-obs targets
  Total rows in features dataset: 2,126
  Saved: data/processed/maharashtra_tomato_features.csv

FEATURE ENGINEERING: WHEAT
  APMC Akola | Other | FAQ: 593 obs -> 590 valid 3-obs targets
  APMC Amarawati | Other | FAQ: 595 obs -> 592 valid 3-obs targets
  APMC Chattrapati Sambhajinagar | Other | FAQ: 558 obs -> 555 valid 3-obs targets
  APMC Dhule | Other | FAQ: 520 obs -> 517 valid 3-obs targets
  APMC Hinganghat | Other | FAQ: 525 obs -> 522 valid 3-obs targets
  APMC Jalana | Other | FAQ: 604 obs -> 601 valid 3-obs targets
  APMC Kalyan | Sharbati | FAQ: 600 obs -> 597 valid 3-obs targets
  APMC Kopargaon | Other | FAQ: 605 obs -> 602 valid 3-obs targets
  APMC Majalgaon | Other | FAQ: 540 obs

  Total rows in features dataset: 12,205
  Saved: data/processed/maharashtra_wheat_features.csv

FEATURE ENGINEERING: COTTON


### 3. Feature Dataset Overview

In [3]:
for crop, df in feat_results.items():
    if df.empty:
        print(f'{crop}: EMPTY'); continue
    print(f'\n{crop}: {len(df):,} rows x {len(df.columns)} columns')
    lag_cols = [c for c in df.columns if 'lag' in c]
    ma_cols = [c for c in df.columns if 'price_ma' in c]
    mom_cols = [c for c in df.columns if 'change' in c]
    cal_cols = [c for c in df.columns if c in ['year','month','day','day_of_week','day_of_year','week_of_year','is_weekend','month_sin','month_cos','day_of_year_sin','day_of_year_cos']]
    tgt_cols = [c for c in df.columns if 'price_next' in c or 'price_after' in c or 'future_' in c]
    print(f'  Lag features ({len(lag_cols)}): {lag_cols}')
    print(f'  MA features  ({len(ma_cols)}): {ma_cols}')
    print(f'  Momentum     ({len(mom_cols)}): {mom_cols}')
    print(f'  Calendar     ({len(cal_cols)}): {cal_cols}')
    print(f'  Targets      ({len(tgt_cols)}): {tgt_cols}')


Tomato: 2,126 rows x 55 columns
  Lag features (8): ['price_order_flag', 'statistical_outlier_flag', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_7', 'price_lag_14', 'price_lag_30']
  MA features  (4): ['price_ma_3', 'price_ma_7', 'price_ma_14', 'price_ma_30']
  Momentum     (10): ['price_change_1', 'price_change_1_pct', 'price_change_3', 'price_change_3_pct', 'price_change_7', 'price_change_7_pct', 'price_change_14', 'price_change_14_pct', 'future_price_change_3', 'future_price_change_7']
  Calendar     (11): ['year', 'month', 'day', 'day_of_week', 'day_of_year', 'week_of_year', 'is_weekend', 'month_sin', 'month_cos', 'day_of_year_sin', 'day_of_year_cos']
  Targets      (5): ['price_next_observation', 'price_after_3_observations', 'price_after_7_observations', 'future_price_change_3', 'future_price_change_7']

Wheat: 12,205 rows x 55 columns
  Lag features (8): ['price_order_flag', 'statistical_outlier_flag', 'price_lag_1', 'price_lag_2', 'price_lag_3', 'price_lag_7', 'pri

### 4. Target Variable Statistics

In [4]:
for crop, df in feat_results.items():
    if df.empty: continue
    print(f'\n{crop} — Target candidate statistics:')
    for tgt in ['price_next_observation','price_after_3_observations','price_after_7_observations']:
        if tgt in df.columns:
            s = df[tgt].dropna()
            print(f'  {tgt}: n={len(s):,} mean=₹{s.mean():.0f} std=₹{s.std():.0f} min=₹{s.min():.0f} max=₹{s.max():.0f}')


Tomato — Target candidate statistics:
  price_next_observation: n=2,122 mean=₹2282 std=₹1145 min=₹300 max=₹7500
  price_after_3_observations: n=2,114 mean=₹2285 std=₹1144 min=₹300 max=₹7500
  price_after_7_observations: n=2,098 mean=₹2293 std=₹1143 min=₹300 max=₹7500

Wheat — Target candidate statistics:
  price_next_observation: n=12,184 mean=₹2902 std=₹470 min=₹1700 max=₹4774
  price_after_3_observations: n=12,142 mean=₹2902 std=₹469 min=₹1700 max=₹4774
  price_after_7_observations: n=12,058 mean=₹2902 std=₹469 min=₹1700 max=₹4774


### 5. Leakage Audit — No Future Target in Feature Space

In [5]:
forbidden = ['price_next_observation','price_after_3_observations','price_after_7_observations',
             'future_price_change_3','future_price_change_7']
input_features = ['Min Price','Max Price','Modal Price',
                  'price_lag_1','price_lag_2','price_lag_3','price_lag_7','price_lag_14','price_lag_30',
                  'price_ma_3','price_ma_7','price_ma_14','price_ma_30',
                  'price_std_7','price_std_14','price_std_30',
                  'price_change_1','price_change_1_pct','price_change_3','price_change_3_pct',
                  'price_change_7','price_change_7_pct','price_change_14','price_change_14_pct',
                  'price_range','price_range_pct',
                  'year','month','day','day_of_week','day_of_year','week_of_year','is_weekend',
                  'month_sin','month_cos','day_of_year_sin','day_of_year_cos',
                  'Market','Variety','Grade']
all_ok = True
for feat in input_features:
    for f in forbidden:
        if feat == f:
            print(f'  LEAKAGE: {feat} is in both input and forbidden!'); all_ok = False
print(f'[{"PASSED" if all_ok else "FAILED"}] Zero future target columns present in input feature set.')

[PASSED] Zero future target columns present in input feature set.


### 6. Series Independence Check

In [6]:
for crop, df in feat_results.items():
    if df.empty: continue
    groups = df.groupby(['Market','Variety','Grade'])
    print(f'{crop}: {groups.ngroups} independent series processed separately — cross-series leakage impossible.')

Tomato: 4 independent series processed separately — cross-series leakage impossible.
Wheat: 21 independent series processed separately — cross-series leakage impossible.


### 7. Missing Values from Initial Lag Windows

In [7]:
for crop, df in feat_results.items():
    if df.empty: continue
    nan_lag1 = df['price_lag_1'].isna().sum()
    nan_lag30 = df['price_lag_30'].isna().sum()
    print(f'  {crop}: price_lag_1 NaN={nan_lag1} | price_lag_30 NaN={nan_lag30} (expected from initial window — no imputation applied)')

  Tomato: price_lag_1 NaN=4 | price_lag_30 NaN=120 (expected from initial window — no imputation applied)
  Wheat: price_lag_1 NaN=21 | price_lag_30 NaN=630 (expected from initial window — no imputation applied)


### 8. Step 11 Completion

In [8]:
print('='*60)
print('STEP 11 — FEATURE ENGINEERING COMPLETE')
print('='*60)
for crop, df in feat_results.items():
    print(f'  {crop}: {len(df):,} rows x {len(df.columns)} columns saved')

STEP 11 — FEATURE ENGINEERING COMPLETE
  Tomato: 2,126 rows x 55 columns saved
  Wheat: 12,205 rows x 55 columns saved
  Cotton: 0 rows x 0 columns saved
